# Bori V2 — SmolLM-135M Korean/English CPT & SFT Pipeline

This notebook implements **Bori V2**, a continuous pre-training (CPT) and vocabulary expansion pipeline to adapt **SmolLM2-135M** to Korean.

### Key Enhancements in Bori V2:
1. **Vocabulary Expansion**: Trains a ByteLevel BPE Korean tokenizer (10,000 tokens) on FineWeb2 Korean and merges it with the base SmolLM tokenizer. This reduces Korean text token fertility by ~2x, making sequence lengths much shorter and training faster.
2. **Subword-based Initialization (EEVE method)**: Initializes new Korean embeddings using the mean of their constituent subword embeddings from the original tokenizer. This preserves base model representations.
3. **Two-Stage Progressive Unfreezing**:
   - **Phase 1a (Embedding Warm-up)**: Backbone frozen, only embeddings and LM head are trained on 100% Korean text (1,000 steps).
   - **Phase 1b (Full CPT)**: Entire model unfrozen, trained on 90% Korean + 10% English replay to prevent catastrophic forgetting (10,000 steps).
4. **Phase 2 (SFT)**: Instruction fine-tuning on Korean instructions (`brandonbaek/konglish-synthetic-instruct`).

Track live progress at [wandb.ai](https://wandb.ai).

### Step 0: Setup Codebase

In [ ]:
import os
import shutil
import zipfile

# Clean up working dir except for existing checkpoints to allow resuming
working_dir = "/kaggle/working"
input_dir = "/kaggle/input"

print("Cleaning working directory...")
for item in os.listdir(working_dir):
    path = os.path.join(working_dir, item)
    if item not in ["checkpoints", "sft_checkpoints", "data"]:
        try:
            if os.path.isdir(path):
                shutil.rmtree(path)
            else:
                os.remove(path)
        except Exception as e:
            print(f"Warning: Could not clean {item}: {e}")

# Copy codebase from kaggle input using a robust recursive search
found_code = False
print(f"Searching for codebase recursively in {input_dir}...")

for root, dirs, files in os.walk(input_dir):
    # Skip folders that are part of checkpoints to avoid mixing up codebase and checkpoints
    if "checkpoint" in root.lower() or "checkpoints" in root.lower():
        continue
    # Option 1: A folder that directly contains the core scripts/configs/src
    if "scripts" in dirs and "src" in dirs:
        print(f"Found codebase folder at: {root}")
        for item in os.listdir(root):
            s = os.path.join(root, item)
            d = os.path.join(working_dir, item)
            if os.path.isdir(s):
                shutil.copytree(s, d, dirs_exist_ok=True)
            else:
                shutil.copy2(s, d)
        found_code = True
        break
        
    # Option 2: A directory named "bori_v2"
    if "bori_v2" in dirs:
        src_dir = os.path.join(root, "bori_v2")
        print(f"Found 'bori_v2' folder at: {src_dir}")
        for item in os.listdir(src_dir):
            s = os.path.join(src_dir, item)
            d = os.path.join(working_dir, item)
            if os.path.isdir(s):
                shutil.copytree(s, d, dirs_exist_ok=True)
            else:
                shutil.copy2(s, d)
        found_code = True
        break

    # Option 3: A zip file containing "bori" or similar
    for f in files:
        if f.endswith(".zip") and ("bori" in f.lower() or "code" in f.lower()):
            zip_path = os.path.join(root, f)
            print(f"Found zipped codebase at: {zip_path}")
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(working_dir)
            
            # If the zip extracted into a subfolder named bori_v2, move its contents up
            extracted_dir = os.path.join(working_dir, "bori_v2")
            if os.path.exists(extracted_dir):
                for extracted_item in os.listdir(extracted_dir):
                    shutil.move(os.path.join(extracted_dir, extracted_item), os.path.join(working_dir, extracted_item))
                os.rmdir(extracted_dir)
            found_code = True
            break
    if found_code:
        break

if found_code:
    print("Codebase successfully copied/extracted.")
else:
    print("⚠️ WARNING: Codebase not found in Kaggle inputs recursively! Creating standard directory structure anyway.")

os.makedirs("/kaggle/working/src", exist_ok=True)
os.makedirs("/kaggle/working/scripts", exist_ok=True)
os.makedirs("/kaggle/working/configs", exist_ok=True)
os.makedirs("/kaggle/working/data", exist_ok=True)




# ──── Robust Direct Checkpoint & Merged Tokenizer Restoration ────
print("Restoring checkpoints and merged tokenizer directly from Kaggle input dataset...")
found_checkpoints = False
found_tokenizer = False

dataset_root = "/kaggle/input/datasets/brandonbaek/bori-v2-checkpoints"
checkpoints_src = os.path.join(dataset_root, "checkpoints")
tokenizer_src = os.path.join(dataset_root, "data/merged_tokenizer")

# 1. Direct restore of checkpoints folder
if os.path.exists(checkpoints_src):
    print(f"Found checkpoints directory at: {checkpoints_src}")
    shutil.copytree(checkpoints_src, "/kaggle/working/checkpoints", dirs_exist_ok=True)
    found_checkpoints = True
    print("✅ Successfully restored CPT checkpoints to /kaggle/working/checkpoints")
else:
    # Search fallback if checkpoints folder is nested differently
    for root, dirs, files in os.walk(dataset_root):
        if "phase_1a" in dirs or "phase_1b" in dirs or any(d.startswith("checkpoint-") for d in dirs):
            # Avoid copying if it's already inside a nested subdirectory we already processed
            if "working/checkpoints" in root:
                continue
            print(f"Found nested checkpoints directory at: {root}")
            shutil.copytree(root, "/kaggle/working/checkpoints", dirs_exist_ok=True)
            found_checkpoints = True
            print("✅ Successfully restored CPT checkpoints to /kaggle/working/checkpoints")
            break

# 2. Direct restore of merged tokenizer from the data/ folder
if os.path.exists(tokenizer_src):
    print(f"Found merged tokenizer directory at: {tokenizer_src}")
    shutil.copytree(tokenizer_src, "/kaggle/working/data/merged_tokenizer", dirs_exist_ok=True)
    found_tokenizer = True
    print("✅ Successfully restored merged tokenizer to /kaggle/working/data/merged_tokenizer")
else:
    # Fallback recursive search for tokenizer inside the dataset
    for root, dirs, files in os.walk(dataset_root):
        if "working/data" in root:
            continue
        if "tokenizer.json" in files and "tokenizer_config.json" in files:
            # Match either a merged tokenizer folder or a checkpoints folder
            if "merged" in root.lower() or "checkpoint" in root.lower() or "phase_1a" in root.lower() or "phase_1b" in root.lower():
                print(f"Found merged tokenizer files at: {root}")
                dest_tok = "/kaggle/working/data/merged_tokenizer"
                os.makedirs(dest_tok, exist_ok=True)
                for f in os.listdir(root):
                    if any(x in f for x in ["tokenizer", "vocab", "special_tokens", "config", "added_tokens"]):
                        shutil.copy2(os.path.join(root, f), os.path.join(dest_tok, f))
                found_tokenizer = True
                print("✅ Successfully restored merged tokenizer to /kaggle/working/data/merged_tokenizer")
                break

# 3. Restore SFT export checkpoints if present
sft_src = os.path.join(dataset_root, "sft_checkpoints")
sft_export_src = os.path.join(dataset_root, "bori_v2_sft_export")
if os.path.exists(sft_src):
    print(f"Found SFT checkpoints folder at: {sft_src}")
    shutil.copytree(sft_src, "/kaggle/working/sft_checkpoints", dirs_exist_ok=True)
    print("✅ Successfully restored SFT checkpoints to /kaggle/working/sft_checkpoints")
elif os.path.exists(sft_export_src):
    print(f"Found SFT export checkpoints folder at: {sft_export_src}")
    shutil.copytree(sft_export_src, "/kaggle/working/sft_checkpoints", dirs_exist_ok=True)
    print("✅ Successfully restored SFT export checkpoints to /kaggle/working/sft_checkpoints")

if not found_checkpoints:
    print("ℹ️ No CPT checkpoints restored from inputs. Starting CPT from scratch.")
if not found_tokenizer:
    print("ℹ️ No merged tokenizer restored from inputs. Will train one if needed.")

print("Working directory structure:")
for root, dirs, files in os.walk(working_dir):
    depth = root.replace(working_dir, "").count(os.sep)
    if depth < 2:
        indent = "  " * depth
        print(f"{indent}📁 {os.path.basename(root)}/")
        for f in files[:5]:
            print(f"{indent}  📄 {f}")

### Step 0b: Install Dependencies

In [ ]:
import sys
!{sys.executable} -m pip install -q "transformers>=4.40.0" "accelerate>=0.29.0" "datasets>=2.18.0" "wandb>=0.16.0" "tokenizers>=0.15.0" rich tqdm gradio

### Step 0c: W&B Login (auto-skips if secret is missing)

In [ ]:
try:
    import wandb
    from kaggle_secrets import UserSecretsClient
    wandb.login(key=UserSecretsClient().get_secret('wandb_api_key'))
    WANDB_PROJECT = 'bori-v2'
    print('W&B logged in. Track at: https://wandb.ai')
except Exception as e:
    print(f'W&B skipped ({e}). Training continues without logging.')
    WANDB_PROJECT = None

### Step 0d: Configure Environment & Checkpoint Recovery

In [ ]:
import torch
import os
import shutil

# Set critical distributed variables to bypass virtualized container limitations on Kaggle
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['NCCL_DISABLE_P2P'] = '1'
os.environ['NCCL_IB_DISABLE'] = '1'

num_gpus = torch.cuda.device_count()
print(f'{num_gpus} GPU(s) detected.')

os.makedirs('configs', exist_ok=True)
accelerate_config = 'configs/accelerate_config_multi.yaml' if num_gpus > 1 else 'configs/accelerate_config_single.yaml'

# Force overwrite accelerate configs with valid keys to avoid stale/buggy settings
print("Overwriting accelerate config files to ensure clean and valid settings...")
with open('configs/accelerate_config_single.yaml', 'w') as f:
    f.write("compute_environment: LOCAL_MACHINE\n")
    f.write("debug: false\n")
    f.write("distributed_type: 'NO'\n")
    f.write("downcast_bf16: 'no'\n")
    f.write("gpu_ids: all\n")
    f.write("machine_rank: 0\n")
    f.write("main_training_function: main\n")
    f.write("mixed_precision: 'no'\n")
    f.write("num_machines: 1\n")
    f.write("num_processes: 1\n")
    f.write("rdzv_backend: static\n")
    f.write("same_network: true\n")
    f.write("use_cpu: false\n")

with open('configs/accelerate_config_multi.yaml', 'w') as f:
    f.write("compute_environment: LOCAL_MACHINE\n")
    f.write("debug: false\n")
    f.write("distributed_type: MULTI_GPU\n")
    f.write("downcast_bf16: 'no'\n")
    f.write("gpu_ids: all\n")
    f.write("machine_rank: 0\n")
    f.write("main_training_function: main\n")
    f.write("mixed_precision: 'no'\n")
    f.write("num_machines: 1\n")
    f.write("num_processes: 2\n")
    f.write("rdzv_backend: static\n")
    f.write("same_network: true\n")
    f.write("tpu_env: []\n")
    f.write("tpu_use_cluster: false\n")
    f.write("tpu_use_sudo: false\n")
    f.write("use_cpu: false\n")

print(f"Using accelerate config: {accelerate_config}")

# Pre-cache model and tokenizer on the single main thread to avoid DDP download race conditions
print("Pre-caching base model and tokenizer...")
from transformers import AutoTokenizer, AutoModelForCausalLM
try:
    AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M")
    AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM2-135M")
    print("✅ Pre-caching complete.")
except Exception as e:
    print(f"⚠️ Warning: Pre-caching failed ({e}). Continuing anyway.")

RUN_NAME = 'bori-v2-135m'
WANDB_RUN_ID_CPT = 'bori-v2-135m-cpt'
    WANDB_RUN_ID_SFT = 'bori-v2-135m-sft-v2'



### Step 1: Tokenizer Vocabulary Expansion
We train a ByteLevel BPE tokenizer on a streaming sample of 500,000 documents from FineWeb2 Korean, then merge it with the base SmolLM tokenizer.
This reduces Korean text token fertility by ~2x.

In [ ]:
import os
import sys
import shutil

python_path = sys.executable

# 1. First, search if a merged tokenizer already exists in our active folder
tokenizer_restored = False
if os.path.exists("data/merged_tokenizer/tokenizer.json"):
    print("✅ Existing merged tokenizer found at data/merged_tokenizer/. Skipping training.")
    tokenizer_restored = True
else:
    # 2. If not, check if we recovered it via our CHECKPOINT_DATASET under /kaggle/working/checkpoints/
    if os.path.exists("/kaggle/working/checkpoints"):
        print("Searching for an existing tokenizer recursively in restored checkpoints...")
        for root, dirs, files in os.walk("/kaggle/working/checkpoints"):
            if "tokenizer.json" in files:
                tokenizer_dir = root
                print(f"Found existing tokenizer files in checkpoint at: {tokenizer_dir}")
                os.makedirs("data/merged_tokenizer", exist_ok=True)
                for f in os.listdir(tokenizer_dir):
                    if any(x in f for x in ["tokenizer", "vocab", "special_tokens", "config", "added_tokens"]):
                        shutil.copy2(os.path.join(tokenizer_dir, f), os.path.join("data/merged_tokenizer", f))
                tokenizer_restored = True
                print("✅ Successfully restored the merged tokenizer from your checkpoint dataset! Skipping training.")
                break

# 3. If no tokenizer was found anywhere, train a new one from scratch
if not tokenizer_restored:
    print("No existing tokenizer found. Training Korean tokenizer from scratch...")
    !{python_path} scripts/train_korean_tokenizer.py \
        --vocab_size 10000 \
        --num_samples 500000 \
        --output_dir data/korean_tokenizer
    
    print("\nMerging tokenizers and running fertility comparison...")
    !{python_path} scripts/merge_tokenizers.py \
        --base_model HuggingFaceTB/SmolLM2-135M \
        --korean_tokenizer_path data/korean_tokenizer \
        --output_dir data/merged_tokenizer

### Step 2: Continuous Pre-training (CPT)
We run the two-stage training script `scripts/cpt.py`:
- **Phase 1a (Embedding Warm-up)**: Trains only embeddings and LM head on Korean (1,000 steps).
- **Phase 1b (Full CPT)**: Unfreezes the backbone and trains the entire model on 90% Korean + 10% English replay (10,000 steps).

In [ ]:
import os
import sys

python_path = sys.executable
wandb_flag = f'--wandb_project {WANDB_PROJECT}' if WANDB_PROJECT else ''
wandb_id_flag = f'--wandb_run_id {WANDB_RUN_ID_CPT}' if WANDB_PROJECT else ''

# ──────────────────────────────────────────────────────────────────────
# Set SKIP_CPT = True if you already finished CPT and only want to run SFT.
# IMPORTANT: For the FIRST run, leave SKIP_PHASE1A = False.
SKIP_CPT = False
# For SUBSEQUENT sessions (resuming Phase 1b after a Kaggle restart),
# set SKIP_PHASE1A = True so it loads from the Phase 1a checkpoint
# and jumps directly to Phase 1b.
# ──────────────────────────────────────────────────────────────────────
# Set SKIP_CPT = True if you already finished CPT and only want to run SFT.
SKIP_PHASE1A = False
SKIP_CPT = False
skip_flag = '--skip_phase1a' if SKIP_PHASE1A else ''

if not SKIP_CPT:
    !PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True {python_path} -m accelerate.commands.launch \
        --config_file {accelerate_config} scripts/cpt.py \
        --model_name HuggingFaceTB/SmolLM2-135M \
        --tokenizer_path data/merged_tokenizer \
        --dataset_name "HuggingFaceFW/fineweb-2:kor_Hang" \
        --replay_dataset "HuggingFaceFW/fineweb-edu" \
        --replay_ratio 0.1 \
        --warmup_lr 1e-3 \
        --warmup_steps_phase1a 1000 \
        --cpt_lr 2e-4 \
        --lr_scheduler_type constant_with_warmup \
        --max_steps 10000 \
        --save_steps 200 \
        --batch_size 2 \
        --grad_accum 4 \
        --max_seq_length 2048 \
        --output_dir /kaggle/working/checkpoints \
        {wandb_flag} {wandb_id_flag} \
        --run_name {RUN_NAME} {skip_flag}
else:
    print("Skipping CPT step as requested.")


### Step 3: Supervised Fine-Tuning (SFT)
We perform instruction tuning using `brandonbaek/konglish-synthetic-instruct` on the continuous pre-trained model.

In [ ]:
import os
import sys

if not os.path.exists('/kaggle/working/checkpoints/phase_1b/config.json'):
    raise ValueError("CPT output model config not found at /kaggle/working/checkpoints/phase_1b/config.json! ")

python_path = sys.executable
wandb_flag = f'--wandb_project {WANDB_PROJECT}' if WANDB_PROJECT else ''
wandb_id_flag = f'--wandb_run_id {WANDB_RUN_ID_SFT}' if WANDB_PROJECT else ''

datasets = "brandonbaek/konglish-synthetic-instruct,jojo0217/korean_safe_conversation,HuggingFaceH4/ultrachat_200k"
probs = "0.6,0.2,0.2"

!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True {python_path} -m accelerate.commands.launch \
    --config_file {accelerate_config} scripts/sft.py \
    --model_path /kaggle/working/checkpoints/phase_1b \
    --tokenizer_path data/merged_tokenizer \
    --dataset_name "{datasets}" \
    --dataset_probs "{probs}" \
    --max_steps 2000 \
    --save_steps 500 \
    --learning_rate 2e-5 \
    --output_dir /kaggle/working/sft_checkpoints \
    {wandb_flag} {wandb_id_flag} \
    --run_name {RUN_NAME}-sft


### Step 4: Verification & Quick Inference Test
Verify that the SFT-trained model can generate valid Korean/Konglish text.

In [ ]:
model_path = "/kaggle/working/sft_checkpoints"
if not os.path.exists(os.path.join(model_path, "config.json")):
    print("SFT not finished or not saved, falling back to CPT checkpoint...")
    model_path = "/kaggle/working/checkpoints/phase_1b"

!{python_path} scripts/test_inference.py \
    --model_path {model_path} \
    --tokenizer_path data/merged_tokenizer

### Step 5: Zip and Export Checkpoints
Export the trained checkpoints so you can download them directly from the Output tab.

In [ ]:
import shutil
import os

print("Zipping checkpoints...")
if os.path.exists("/kaggle/working/checkpoints/phase_1b"):
    shutil.make_archive("/kaggle/working/bori_v2_cpt_export", "zip", "/kaggle/working/checkpoints/phase_1b")
    print("CPT Model Exported.")

if os.path.exists("/kaggle/working/sft_checkpoints"):
    shutil.make_archive("/kaggle/working/bori_v2_sft_export", "zip", "/kaggle/working/sft_checkpoints")
    print("SFT Model Exported.")

print("All tasks completed! Download the zip files from the Output tab.")